# 左右方向与平衡选项：小规模协议诊断

先跑默认 Qwen-27B；完成后换全新 GPU 运行时，仅把 `MODEL_KEY` 改为 `gemma12b`，从头运行。

[实验设置与运行说明](../../../docs/experiment_notes/90_followup_visual_and_relation_use/42f53df649d0af6a.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready.')


In [ ]:
%%cell_progress 1/13 Mount Drive and locate project
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/13 Choose model and source development outputs
MODEL_KEY = 'qwen27b'  # Run Qwen first, then gemma12b in a FRESH GPU runtime.
RUN_ID = 'main'
SOURCE_RUN_ID = 'main'  # The previous information-support development run.
SEED = 2026091802
SOURCE_ZIPS = {'qwen27b': None, 'gemma12b': None}
# None locates the previous output ZIP in its original Drive result folder.
# If needed, replace None with an exact ZIP path. Do not set PHASE to validation.


In [ ]:
%%cell_progress 3/13 Install inference dependencies
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'Pillow==11.3.0',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 4/13 Load frozen diagnostic design
import types, sys
_base_module=types.ModuleType("visual_information_support_core")
sys.modules[_base_module.__name__]=_base_module
exec(compile('"""CPU design, rendering, prompts and family-paired analysis for information support."""\nfrom __future__ import annotations\n\nimport hashlib\nimport itertools\nimport json\nfrom pathlib import Path\nfrom collections import Counter\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFont\n\nSCHEMA = \'visual-information-support-v1\'\nCONDITIONS = (\'image\', \'image_weight\', \'image_distance\', \'image_full\', \'text_full\', \'legacy_image\')\nSTRATA = (\'congruent\', \'balanced\', \'equal_weight_distance\', \'weight_conflict\', \'binding_swap\')\nLABELS = (\'A\', \'B\', \'C\')\nSEMANTICS = (\'left\', \'balanced\', \'right\')\nSEMANTIC_TEXT = {\'left\': \'left side\', \'balanced\': \'equal\', \'right\': \'right side\'}\nRULE_TEXT = (\'Each square is one unit of weight. Distance is measured in tick \'\n             \'intervals from the center pivot. Torque equals weight times distance. \'\n             \'Total torque on a side is the sum over stacks on that side.\')\nREPLY_TEXT = (\'Do not explain your reasoning. Reply with exactly one option letter \'\n              \'and nothing else. Valid replies are: A, B, C.\')\nMODEL_SPECS = {\n    \'gemma12b\': dict(model_id=\'google/gemma-4-12B-it\',\n                    revision=\'707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7\'),\n    \'qwen27b\': dict(model_id=\'Qwen/Qwen3.5-27B\',\n                   revision=\'373612cb4f9288fcf550b526d86d85febedd9801\'),\n}\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                    allow_nan=False).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef save_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef freeze(path, obj):\n    if Path(path).exists():\n        assert read_json(path) == obj, f\'Frozen configuration changed: {path}. Use a new RUN_ID.\'\n    else:\n        save_json(path, obj)\n    return digest(obj)\n\n\ndef signature(stacks):\n    direct = tuple(sorted((s[\'side\'], int(s[\'distance\']), int(s[\'units\'])) for s in stacks))\n    mirror = tuple(sorted((\'right\' if s == \'left\' else \'left\', d, w) for s, d, w in direct))\n    return min(direct, mirror)\n\n\ndef geometries_in(obj):\n    found = set()\n    if isinstance(obj, dict):\n        if isinstance(obj.get(\'stacks\'), list) and obj[\'stacks\']:\n            stacks = obj[\'stacks\']\n            if all(isinstance(s, dict) and {\'side\', \'distance\', \'units\'} <= s.keys() for s in stacks):\n                found.add(signature(stacks))\n        for key, value in obj.items():\n            if key == \'signatures\' and isinstance(value, list):\n                for sig in value:\n                    found.add(signature([dict(side=s, distance=d, units=w) for s, d, w in sig]))\n            elif key != \'stacks\':\n                found.update(geometries_in(value))\n    elif isinstance(obj, list):\n        for item in obj:\n            found.update(geometries_in(item))\n    return found\n\n\ndef load_history(project, run_root, source_run=\'main\', extra_files=(), progress=print):\n    """Read only explicit old supports/snapshots and shallow dataset manifests; no recursive Drive walk."""\n    project, run_root = Path(project), Path(run_root)\n    lock = run_root / \'historical_exclusion.json\'\n    if lock.exists():\n        old = read_json(lock)\n        assert old[\'source_run\'] == source_run and old[\'extra_files\'] == [str(p) for p in extra_files]\n        for name, sha in old[\'files\'].items():\n            assert file_hash(name) == sha, \'Historical input changed: \' + name\n        progress(f"Reusing frozen exclusions: {len(old[\'signatures\'])} geometries; no directory scan.")\n        return geometries_in(old)\n    paths = set()\n    data = project / \'data\'\n    if data.exists():\n        paths.update(data.glob(\'*/scenes.jsonl\'))\n        if (data / \'scenes.jsonl\').is_file():\n            paths.add(data / \'scenes.jsonl\')\n    results = project / \'results\'\n    if results.exists():\n        # Only top-level experiment directories and this known run, never activation shards.\n        for experiment in results.iterdir():\n            if experiment.is_dir() and experiment.resolve() != run_root.parent.resolve():\n                for folder in (experiment, experiment / source_run):\n                    for name in (\'historical_geometry_snapshot.json\', \'support.json\', \'scenes.jsonl\'):\n                        if (folder / name).is_file():\n                            paths.add(folder / name)\n    for name in extra_files:\n        p = Path(name)\n        assert p.is_file(), f\'EXTRA_HISTORY_FILES must name files: {p}\'\n        paths.add(p)\n    assert paths, \'No historical manifests found. Restore old support/snapshot files, or set EXTRA_HISTORY_FILES.\'\n    signatures, hashes = set(), {}\n    for i, path in enumerate(sorted(paths)):\n        progress(f\'Read historical manifest {i+1}/{len(paths)}: {path.name}\')\n        h = hashlib.sha256()\n        with path.open(\'rb\') as stream:\n            if path.suffix == \'.jsonl\':\n                for line in stream:\n                    h.update(line)\n                    if line.strip():\n                        signatures.update(geometries_in(json.loads(line.decode(\'utf-8-sig\'))))\n            else:\n                raw = stream.read()\n                h.update(raw)\n                signatures.update(geometries_in(json.loads(raw.decode(\'utf-8-sig\'))))\n        hashes[str(path)] = h.hexdigest()\n    assert signatures, \'No usable historical geometry was found in the selected manifests.\'\n    save_json(lock, dict(source_run=source_run, extra_files=[str(p) for p in extra_files],\n                         files=hashes, signatures=[[list(t) for t in sig] for sig in sorted(signatures)],\n                         scope=\'Listed manifests/snapshots only; unseen ZIPs or unlisted runs are not certified disjoint.\'))\n    return signatures\n\n\ndef scene_features(stacks):\n    w = {s: sum(x[\'units\'] for x in stacks if x[\'side\'] == s) for s in (\'left\', \'right\')}\n    torque = {s: sum(x[\'units\'] * x[\'distance\'] for x in stacks if x[\'side\'] == s)\n              for s in (\'left\', \'right\')}\n    delta = torque[\'left\'] - torque[\'right\']\n    weight = w[\'left\'] - w[\'right\']\n    outcome = lambda x: \'left\' if x > 0 else \'right\' if x < 0 else \'balanced\'\n    return dict(left_torque=torque[\'left\'], right_torque=torque[\'right\'], net_torque=delta,\n                weight_score=weight, outcome=outcome(delta), weight_outcome=outcome(weight))\n\n\ndef allocation(n):\n    assert isinstance(n, int) and n >= 32 and n % 8 == 0\n    return dict(congruent=n//8, balanced=n//8, equal_weight_distance=n//4,\n                weight_conflict=n//4, binding_swap=n//4)\n\n\ndef _candidate(rng, stratum):\n    dl, dr = [sorted(rng.choice(np.arange(1, 7), 2, replace=False).tolist()) for _ in range(2)]\n    wl, wr = rng.integers(1, 7, size=(2, 2)).tolist()\n    if stratum == \'equal_weight_distance\':\n        wr = wl.copy()\n        if not (all(a > b for a, b in zip(dl, dr)) or all(a < b for a, b in zip(dl, dr))):\n            return None\n    stacks = [dict(side=side, distance=d, units=w) for side, distances, weights in\n              [(\'left\', dl, wl), (\'right\', dr, wr)] for d, w in zip(distances, weights)]\n    f = scene_features(stacks)\n    if stratum == \'congruent\':\n        distance_delta = sum(dl) - sum(dr)\n        if not (f[\'weight_score\'] * f[\'net_torque\'] > 0 and distance_delta * f[\'net_torque\'] > 0):\n            return None\n    elif stratum == \'balanced\' and f[\'net_torque\'] != 0:\n        return None\n    elif stratum == \'weight_conflict\' and f[\'weight_score\'] * f[\'net_torque\'] >= 0:\n        return None\n    roles = {\'base\': stacks}\n    if stratum == \'binding_swap\':\n        swapped = [dict(s) for s in stacks]\n        swapped[0][\'units\'], swapped[1][\'units\'] = swapped[1][\'units\'], swapped[0][\'units\']\n        if f[\'net_torque\'] * scene_features(swapped)[\'net_torque\'] >= 0:\n            return None\n        roles[\'swapped\'] = swapped\n    return roles\n\n\ndef generate_families(n, seed, excluded, phase):\n    rng = np.random.default_rng(seed)\n    used = set(excluded)\n    families = []\n    for stratum, count in allocation(n).items():\n        accepted = 0\n        for attempt in range(300000):\n            roles = _candidate(rng, stratum)\n            if roles is None:\n                continue\n            sigs = {signature(s) for s in roles.values()}\n            if len(sigs) != len(roles) or sigs & used:\n                continue\n            fid = \'vis_\' + digest([SCHEMA, phase, stratum, sorted(sigs)])[:16]\n            slot_ids = np.random.default_rng(int(digest([fid, \'ids\'])[:16], 16)).permutation(4).tolist()\n            records = []\n            for role, stacks in roles.items():\n                for mirror in (False, True):\n                    actual = [dict(s, stack_id=f\'S{slot_ids[i]+1}\',\n                                   side=(\'right\' if s[\'side\'] == \'left\' else \'left\') if mirror else s[\'side\'])\n                              for i, s in enumerate(stacks)]\n                    sid = f\'{fid}__{role}__m{int(mirror)}\'\n                    records.append(dict(scene_id=sid, role=role, mirror=mirror,\n                                        stacks=actual, features=scene_features(actual)))\n            families.append(dict(family_id=fid, stratum=stratum, phase=phase, scenes=records))\n            used.update(sigs)\n            accepted += 1\n            if accepted == count:\n                break\n        assert accepted == count, f\'Insufficient unused {stratum} geometries: {accepted}/{count}; do not relax exclusions.\'\n    families.sort(key=lambda f: digest([seed, f[\'family_id\']]))\n    validate_families(families, excluded)\n    return families\n\n\ndef validate_families(families, excluded=()):\n    used = set(excluded)\n    assert len({f[\'family_id\'] for f in families}) == len(families)\n    for family in families:\n        base = [s for s in family[\'scenes\'] if not s[\'mirror\']]\n        sigs = {signature(s[\'stacks\']) for s in base}\n        assert len(sigs) == len(base) and not sigs & used\n        used.update(sigs)\n        for scene in family[\'scenes\']:\n            assert scene_features(scene[\'stacks\']) == scene[\'features\']\n            assert len(scene[\'stacks\']) == 4\n            assert len({(s[\'side\'], s[\'distance\']) for s in scene[\'stacks\']}) == 4\n            assert len({s[\'stack_id\'] for s in scene[\'stacks\']}) == 4\n            assert all(1 <= s[\'units\'] <= 6 and 1 <= s[\'distance\'] <= 6 for s in scene[\'stacks\'])\n        if family[\'stratum\'] == \'binding_swap\':\n            a, b = base\n            assert a[\'features\'][\'net_torque\'] * b[\'features\'][\'net_torque\'] < 0\n            for side in (\'left\', \'right\'):\n                aa = [s for s in a[\'stacks\'] if s[\'side\'] == side]\n                bb = [s for s in b[\'stacks\'] if s[\'side\'] == side]\n                assert sorted(s[\'units\'] for s in aa) == sorted(s[\'units\'] for s in bb)\n                assert sorted(s[\'distance\'] for s in aa) == sorted(s[\'distance\'] for s in bb)\n            assert {s[\'stack_id\']: (s[\'side\'], s[\'distance\']) for s in a[\'stacks\']} == {\n                s[\'stack_id\']: (s[\'side\'], s[\'distance\']) for s in b[\'stacks\']}\n\n\ndef mappings_for(family_id):\n    offset = int(digest([family_id, \'mapping\'])[:8], 16) % 3\n    return [tuple(SEMANTICS[(i + k + offset) % 3] for i in range(3)) for k in range(3)]\n\n\ndef public_fields(scene, condition):\n    assert condition in CONDITIONS\n    full = condition in (\'image_full\', \'text_full\')\n    return [dict(stack_id=s[\'stack_id\'], side=s[\'side\'],\n                 weight=s[\'units\'] if full or condition == \'image_weight\' else \'not provided\',\n                 distance=s[\'distance\'] if full or condition == \'image_distance\' else \'not provided\')\n            for s in sorted(scene[\'stacks\'], key=lambda s: s[\'stack_id\'])]\n\n\ndef prompt_for(scene, condition, semantics):\n    options = \'\\n\'.join(f\'{label}. {SEMANTIC_TEXT[meaning]}\' for label, meaning in zip(LABELS, semantics))\n    if condition == \'legacy_image\':\n        parts = [\'Answer this question using only the supplied balance-scale scene.\', RULE_TEXT]\n    else:\n        fields = public_fields(scene, condition)\n        table = \'\\n\'.join(f"{r[\'stack_id\']}: side={r[\'side\']}; weight={r[\'weight\']}; distance={r[\'distance\']}" for r in fields)\n        parts = [\'Answer this question using the supplied scene information.\', RULE_TEXT,\n                 \'Stack IDs identify positions; they do not encode weight or distance. \'\n                 \'Use any supplied image and provided values. A field marked not provided supplies no numeric value.\',\n                 \'Provided stack information:\\n\' + table]\n    parts += [\'Question: Which way will the balance scale tip?\', \'Options:\\n\' + options, REPLY_TEXT]\n    return \'\\n\\n\'.join(parts)\n\n\ndef trial_specs(families):\n    for family in families:\n        for scene in family[\'scenes\']:\n            for mapping_index, semantics in enumerate(mappings_for(family[\'family_id\'])):\n                for condition in CONDITIONS:\n                    eid = scene[\'scene_id\'] + f\'__map{mapping_index}__{condition}\'\n                    yield dict(trial_id=eid, family_id=family[\'family_id\'], stratum=family[\'stratum\'],\n                               scene_id=scene[\'scene_id\'], role=scene[\'role\'], mirror=scene[\'mirror\'],\n                               mapping_index=mapping_index, semantics=list(semantics), condition=condition,\n                               gold_semantic=scene[\'features\'][\'outcome\'],\n                               gold_label=LABELS[semantics.index(scene[\'features\'][\'outcome\'])],\n                               weight_outcome=scene[\'features\'][\'weight_outcome\'])\n\n\ndef render_images(families, image_dir):\n    from visual_torque_toolkit.render import RenderStyle, render_scene\n    from visual_torque_toolkit.world import Scene, Side, Stack\n    image_dir = Path(image_dir)\n    image_dir.mkdir(parents=True, exist_ok=True)\n    lock = image_dir.parent / \'image_manifest.json\'\n    if lock.exists():\n        manifest = read_json(lock)\n        expected = {s[\'scene_id\'] + suffix for f in families for s in f[\'scenes\']\n                    for suffix in (\'__plain.png\', \'__ids.png\')}\n        assert set(manifest) == expected\n        for name, entry in manifest.items():\n            assert file_hash(image_dir / name) == entry[\'sha256\'], \'Frozen image missing/changed: \' + name\n        return manifest\n    style = RenderStyle()\n    # Built-in scalable Pillow font travels with the pinned Pillow package.\n    font = ImageFont.load_default(size=18)\n    manifest = {}\n    for family in families:\n        for record in family[\'scenes\']:\n            scene = Scene(record[\'scene_id\'], tuple(Stack(s[\'stack_id\'], Side(s[\'side\']), s[\'distance\'], s[\'units\'])\n                                                   for s in record[\'stacks\']))\n            plain = image_dir / (record[\'scene_id\'] + \'__plain.png\')\n            render_scene(scene, plain, style)\n            with Image.open(plain) as source:\n                labeled = source.convert(\'RGB\')\n            draw = ImageDraw.Draw(labeled)\n            spacing = int(style.width * .38) / style.ticks_per_side\n            boxes = []\n            for s in scene.stacks:\n                x = round(style.width // 2 + (-1 if s.side is Side.LEFT else 1) * s.distance * spacing)\n                y = style.beam_y + 25\n                box = draw.textbbox((x, y), s.stack_id, font=font, anchor=\'mt\')\n                assert 0 <= box[0] < box[2] < style.width and 0 <= box[1] < box[3] < style.height\n                assert all(box[2] <= b[0] or box[0] >= b[2] or box[3] <= b[1] or box[1] >= b[3] for b in boxes)\n                boxes.append(box)\n                draw.text((x, y), s.stack_id, font=font, anchor=\'mt\', fill=style.ink)\n            labeled.save(image_dir / (record[\'scene_id\'] + \'__ids.png\'))\n            for name in (plain.name, record[\'scene_id\'] + \'__ids.png\'):\n                manifest[name] = dict(sha256=file_hash(image_dir / name), width=style.width, height=style.height)\n    save_json(lock, manifest)\n    return manifest\n\n\ndef generation_audit_ids(families):\n    selected = set()\n    for stratum in STRATA:\n        choices = sorted((f for f in families if f[\'stratum\'] == stratum), key=lambda f: digest([\'generation\', f[\'family_id\']]))[:2]\n        for f in choices:\n            scene = next(s for s in f[\'scenes\'] if s[\'role\'] == \'base\' and not s[\'mirror\'])\n            k = int(digest([\'audit-map\', f[\'family_id\']])[:8], 16) % 3\n            selected.update(scene[\'scene_id\'] + f\'__map{k}__{c}\' for c in CONDITIONS)\n    return selected\n\n\ndef strict_label(text):\n    value = text.strip()\n    return value if value in LABELS else None\n\n\ndef holm(p_values):\n    p = np.asarray(p_values, dtype=float)\n    order = np.argsort(p)\n    adjusted = np.empty(len(p))\n    previous = 0.\n    for rank, i in enumerate(order):\n        previous = max(previous, min(1., p[i] * (len(p) - rank)))\n        adjusted[i] = previous\n    return adjusted\n\n\ndef paired_stats(values, key, replicates=10000):\n    x = np.asarray(values, dtype=float)\n    assert x.ndim == 1 and len(x) and np.isfinite(x).all()\n    rng = np.random.default_rng(int(digest(key)[:16], 16))\n    draws = np.concatenate([rng.choice(x, (min(250, replicates-i), len(x)), replace=True).mean(axis=1)\n                            for i in range(0, replicates, 250)])\n    nonzero = x[x != 0]\n    observed = abs(float(x.sum()))\n    if len(nonzero) <= 16:\n        permuted = np.array([sum(a*b for a, b in zip(signs, nonzero))\n                             for signs in itertools.product((-1., 1.), repeat=len(nonzero))])\n        p = float(np.mean(np.abs(permuted) >= observed - 1e-12))\n    else:\n        extreme = 0\n        for i in range(0, replicates, 250):\n            signs = rng.choice((-1., 1.), size=(min(250, replicates-i), len(nonzero)))\n            extreme += int((np.abs(signs @ nonzero) >= observed - 1e-12).sum())\n        p = (extreme + 1) / (replicates + 1)\n    return dict(n_families=int(len(x)), mean=float(x.mean()),\n                lower=float(np.quantile(draws, .025)), upper=float(np.quantile(draws, .975)), p_value=p)\n\n\ndef analyze_rows(rows, families, outdir, model_key, phase, replicates=10000):\n    outdir = Path(outdir)\n    outdir.mkdir(parents=True, exist_ok=True)\n    df = pd.DataFrame(rows)\n    expected = {s[\'trial_id\'] for s in trial_specs(families)}\n    assert df.trial_id.is_unique and set(df.trial_id) == expected, \'Analysis requires every planned trial.\'\n    df.to_json(outdir / \'trial_results.jsonl\', orient=\'records\', lines=True, force_ascii=False)\n    metrics = [\'correct\', \'gold_margin\', \'predicted_balance\', \'wrong_follows_weight\']\n    fam = df.groupby([\'family_id\', \'stratum\', \'condition\'])[metrics].mean().reset_index()\n    fam.to_csv(outdir / \'family_metrics.csv\', index=False)\n    summaries = []\n    for scope in (\'all\', *STRATA):\n        subset = fam if scope == \'all\' else fam[fam.stratum == scope]\n        for condition, group in subset.groupby(\'condition\'):\n            row = dict(model=model_key, phase=phase, scope=scope, condition=condition,\n                       **paired_stats(group.correct, [model_key, phase, scope, condition], replicates))\n            row.pop(\'p_value\')\n            row.update(gold_margin=float(group.gold_margin.mean()),\n                       balance_answer_rate=float(group.predicted_balance.mean()),\n                       wrong_weight_response_rate=float(group.wrong_follows_weight.mean()))\n            summaries.append(row)\n    summary = pd.DataFrame(summaries)\n    summary.to_csv(outdir / \'accuracy_by_condition.csv\', index=False)\n    pair_rows = []\n    subset = df[df.stratum == \'binding_swap\']\n    for keys, group in subset.groupby([\'family_id\', \'condition\', \'mirror\', \'mapping_index\']):\n        assert set(group.role) == {\'base\', \'swapped\'} and len(group) == 2\n        a, b = [group[group.role == role].iloc[0] for role in (\'base\', \'swapped\')]\n        pair_rows.append(dict(family_id=keys[0], condition=keys[1], mirror=keys[2], mapping_index=keys[3],\n                              both_correct=int(a.correct and b.correct),\n                              answer_changed=int(a.predicted_semantic != b.predicted_semantic)))\n    pairs = pd.DataFrame(pair_rows)\n    pairs.to_csv(outdir / \'binding_pair_trials.csv\', index=False)\n    pair_fam = pairs.groupby([\'family_id\', \'condition\'])[[\'both_correct\', \'answer_changed\']].mean().reset_index()\n    pair_fam.to_csv(outdir / \'binding_pair_families.csv\', index=False)\n    definitions = [\n        (\'weight_support_all\', \'all\', [(\'image_weight\', 1), (\'image\', -1)], \'correct\'),\n        (\'distance_support_all\', \'all\', [(\'image_distance\', 1), (\'image\', -1)], \'correct\'),\n        (\'full_support_all\', \'all\', [(\'image_full\', 1), (\'image\', -1)], \'correct\'),\n        (\'joint_support_interaction\', \'all\', [(\'image_full\', 1), (\'image_weight\', -1), (\'image_distance\', -1), (\'image\', 1)], \'correct\'),\n        (\'distance_equal_weight\', \'equal_weight_distance\', [(\'image_distance\', 1), (\'image\', -1)], \'correct\'),\n        (\'full_weight_conflict\', \'weight_conflict\', [(\'image_full\', 1), (\'image\', -1)], \'correct\'),\n        (\'full_binding_pair\', \'binding_swap\', [(\'image_full\', 1), (\'image\', -1)], \'both_correct\'),\n    ]\n    comparisons = []\n    for name, scope, coefficients, metric in definitions:\n        source = pair_fam if metric == \'both_correct\' else (fam if scope == \'all\' else fam[fam.stratum == scope])\n        pivot = source.pivot(index=\'family_id\', columns=\'condition\', values=metric)\n        delta = sum(coef * pivot[condition] for condition, coef in coefficients)\n        comparisons.append(dict(model=model_key, phase=phase, comparison=name, metric=metric, scope=scope,\n                                **paired_stats(delta.to_numpy(), [model_key, phase, name], replicates)))\n    tests = pd.DataFrame(comparisons)\n    tests[\'holm_within_model\'] = holm(tests.p_value)\n    tests[\'inference_status\'] = \'development_descriptive_only\' if phase == \'development\' else \'pending_joint_14_test_correction\'\n    tests.to_csv(outdir / \'primary_comparisons.csv\', index=False)\n    event_rows = []\n    keys = [\'family_id\', \'scene_id\', \'role\', \'mirror\', \'mapping_index\']\n    base = df[df.condition == \'image\'][keys + [\'correct\']].rename(columns={\'correct\': \'baseline_correct\'})\n    for c in CONDITIONS:\n        joined = df[df.condition == c].merge(base, on=keys, validate=\'one_to_one\')\n        joined[\'repaired\'] = (joined.baseline_correct == 0) & (joined.correct == 1)\n        joined[\'harmed\'] = (joined.baseline_correct == 1) & (joined.correct == 0)\n        joined[\'original_error\'] = joined.baseline_correct == 0\n        joined[\'original_correct\'] = joined.baseline_correct == 1\n        for scope in (\'all\', *STRATA):\n            g = joined if scope == \'all\' else joined[joined.stratum == scope]\n            event_rows.append(dict(condition=c, scope=scope, n_trials=len(g),\n                                   repairs=int(g.repaired.sum()), harms=int(g.harmed.sum()),\n                                   original_errors=int(g.original_error.sum()), original_correct=int(g.original_correct.sum()),\n                                   repairs_per_original_error=float(g.repaired.sum()/g.original_error.sum()) if g.original_error.sum() else None,\n                                   harms_per_original_correct=float(g.harmed.sum()/g.original_correct.sum()) if g.original_correct.sum() else None))\n    pd.DataFrame(event_rows).to_csv(outdir / \'repair_harm_counts.csv\', index=False)\n    audits = df[df.generation_audited == True]\n    audit_summary = []\n    for condition, group in audits.groupby(\'condition\'):\n        valid = group.generated_label.isin(LABELS)\n        agreed = valid & (group.generated_label == group.predicted_label)\n        audit_summary.append(dict(condition=condition, n=len(group), valid=int(valid.sum()),\n                                  agrees=int(agreed.sum()), agreement_all=float(agreed.mean()),\n                                  agreement_valid=float(agreed.sum()/valid.sum()) if valid.sum() else None))\n    pd.DataFrame(audit_summary).to_csv(outdir / \'generation_scoring_agreement.csv\', index=False)\n    return summary, tests\n',"<embedded-parent-core>","exec"),_base_module.__dict__)
exec(compile('"""Small, descriptive protocol diagnostic; does not modify the parent study."""\nfrom __future__ import annotations\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw\nimport visual_information_support_core as base\nfrom visual_information_support_core import (\n    digest, file_hash, read_json, save_json, freeze, MODEL_SPECS, LABELS,\n    SEMANTIC_TEXT, strict_label,\n)\n\nSCHEMA = \'direction-balance-diagnostic-v1\'\nPHASE = \'diagnostic\'\nCONDITIONS = (\'legacy_image\', \'image\', \'image_full\', \'text_full\')\nWORDINGS = (\'original\', \'explicit_balance\', \'neutral_question\')\nGROUPS = (\'weight_direction\', \'distance_direction\', \'symmetric_balance\', \'compensated_balance\')\n\n\ndef build_controls():\n    # Deliberate positive controls; familiar geometry is allowed, not a held-out sample.\n    definitions = [\n        (\'weight_direction\', [(\'left\',2,3),(\'right\',2,1)]),\n        (\'weight_direction\', [(\'left\',4,6),(\'right\',4,2)]),\n        (\'distance_direction\', [(\'left\',4,2),(\'right\',1,2)]),\n        (\'distance_direction\', [(\'left\',6,3),(\'right\',2,3)]),\n        (\'symmetric_balance\', [(\'left\',2,2),(\'right\',2,2)]),\n        (\'symmetric_balance\', [(\'left\',2,1),(\'left\',4,2),(\'right\',2,1),(\'right\',4,2)]),\n        (\'compensated_balance\', [(\'left\',4,1),(\'right\',2,2)]),\n        (\'compensated_balance\', [(\'left\',1,1),(\'left\',3,2),(\'right\',1,2),(\'right\',5,1)]),\n    ]\n    families=[]\n    for i,(group,tuples) in enumerate(definitions):\n        fid=f\'db_{i:02d}_{group}\'\n        ids=np.random.default_rng(int(digest([SCHEMA,fid])[:16],16)).permutation(len(tuples))+1\n        scenes=[]\n        for mirror in (False,True):\n            stacks=[dict(side=(\'right\' if side==\'left\' else \'left\') if mirror else side,\n                         distance=d,units=w,stack_id=f\'S{ids[j]}\') for j,(side,d,w) in enumerate(tuples)]\n            scenes.append(dict(scene_id=f\'{fid}__base__m{int(mirror)}\',role=\'base\',mirror=mirror,\n                               stacks=stacks,features=base.scene_features(stacks)))\n        families.append(dict(family_id=fid,stratum=group,scope=\'simple_controls\',scenes=scenes))\n    return families\n\n\ndef diagnostic_prompt(scene, condition, semantics, wording):\n    prompt=base.prompt_for(scene,condition,semantics)\n    if wording!=\'original\':\n        label=LABELS[semantics.index(\'balanced\')]\n        prompt=prompt.replace(f\'{label}. equal\',f\'{label}. balanced (neither side tips)\')\n    if wording==\'neutral_question\':\n        prompt=prompt.replace(\'Question: Which way will the balance scale tip?\',\n                              \'Question: What happens to the balance scale?\')\n    assert wording in WORDINGS\n    return prompt\n\n\ndef load_archives(paths, expected_implementation):\n    snapshots={}\n    pictures={}\n    common=None\n    families=None\n    for key in MODEL_SPECS:\n        path=Path(paths[key])\n        assert path.is_file(), f\'Missing prior development ZIP for {key}: {path}. Set SOURCE_ZIPS to its exact path.\'\n        with zipfile.ZipFile(path) as z:\n            manifest=json.loads(z.read(\'export_manifest.json\'))\n            assert manifest[\'status\']==\'complete\' and manifest[\'phase\']==\'development\' and manifest[\'model\']==key\n            assert set(z.namelist())==set(manifest[\'files\'])|{\'export_manifest.json\'}\n            for name,sha in manifest[\'files\'].items():\n                assert hashlib.sha256(z.read(name)).hexdigest()==sha, \'Source ZIP hash mismatch: \'+name\n            support=json.loads(z.read(\'shared/development/support.json\'))\n            old_inputs=json.loads(z.read(\'shared/development/inputs.json\'))\n            selected=[f for f in support[\'families\'] if f[\'stratum\']==\'balanced\']\n            assert len(selected)==4\n            old_scene_ids={s[\'scene_id\'] for f in selected for s in f[\'scenes\']}\n            root=f\'models/{key}/development/\'\n            runtime=json.loads(z.read(root+\'runtime_identity.json\'))\n            assert runtime[\'implementation\']==expected_implementation, \'This diagnostic expects the reviewed information-support development protocol.\'\n            assert runtime[\'model_metadata\'][\'resolved_revision\']==MODEL_SPECS[key][\'revision\']\n            identity={name:manifest[\'files\'][name] for name in (\'protocol.json\',\'shared/development/support.json\',\'shared/development/inputs.json\')}\n            if common is None:\n                common=identity\n                families=[dict(f,scope=\'archived_balance_replay\',stratum=\'archived_balance\') for f in selected]\n            else:\n                assert identity==common, \'The two development ZIPs used different shared inputs.\'\n            old_rows={}\n            for name in z.namelist():\n                if name.startswith(root+\'families/\') and name.endswith(\'.json\'):\n                    payload=json.loads(z.read(name))\n                    assert digest(payload[\'rows\'])==payload[\'rows_sha256\']\n                    old_rows.update({r[\'trial_id\']:r for r in payload[\'rows\'] if r[\'scene_id\'] in old_scene_ids})\n            chosen=[]\n            for f in selected:\n                # Chosen using identity only, not original outcomes or scores.\n                mapping=int(digest([\'replay-map\',f[\'family_id\']])[:8],16)%3\n                chosen.extend(inp for inp in old_inputs if inp[\'family_id\']==f[\'family_id\']\n                              and inp[\'mapping_index\']==mapping and inp[\'condition\'] in (\'legacy_image\',\'image_full\',\'text_full\'))\n            assert len(chosen)==24\n            for inp in chosen:\n                assert inp[\'trial_id\'] in old_rows\n                if inp[\'image_filename\']:\n                    raw=z.read(\'shared/development/images/\'+inp[\'image_filename\'])\n                    assert hashlib.sha256(raw).hexdigest()==inp[\'image_sha256\']\n                    if inp[\'image_filename\'] in pictures:\n                        assert pictures[inp[\'image_filename\']]==raw\n                    pictures[inp[\'image_filename\']]=raw\n            snapshots[key]=dict(zip_name=path.name,zip_sha256=file_hash(path),shared_identity=identity,\n                runtime=runtime,inputs=chosen,rows={x[\'trial_id\']:old_rows[x[\'trial_id\']] for x in chosen})\n    assert snapshots[\'gemma12b\'][\'inputs\']==snapshots[\'qwen27b\'][\'inputs\']\n    return families,snapshots,pictures\n\n\ndef prepare_study(shared_root, paths, expected_implementation):\n    shared_root=Path(shared_root)\n    controls=build_controls()\n    archived,snapshots,pictures=load_archives(paths,expected_implementation)\n    freeze(shared_root/\'source_snapshots.json\',snapshots)\n    families=controls+archived\n    support=dict(schema=SCHEMA,scope=\'descriptive_diagnostic_only\',families=families,\n                 simple_families=8,archived_families=4,archive_overlap=\'intentional replay\',\n                 generalization_claim=False)\n    freeze(shared_root/\'support.json\',support)\n    image_dir=shared_root/\'images\'\n    # The base renderer supports two- and four-stack scenes.\n    # Keep its manifest in a separate folder to avoid mixing archive originals.\n    control_dir=shared_root/\'control_render/images\'\n    control_manifest=base.render_images(controls,control_dir)\n    image_dir.mkdir(parents=True,exist_ok=True)\n    for name in control_manifest:\n        raw=(control_dir/name).read_bytes()\n        target=image_dir/name\n        if target.exists():\n            assert target.read_bytes()==raw\n        else:\n            target.write_bytes(raw)\n    for name,raw in pictures.items():\n        target=image_dir/name\n        if target.exists():\n            assert target.read_bytes()==raw\n        else:\n            target.write_bytes(raw)\n    inputs=[]\n    first_per_group={group:next(f[\'family_id\'] for f in controls if f[\'stratum\']==group) for group in GROUPS}\n    for f in controls:\n        for s in f[\'scenes\']:\n            for mapping,semantics in enumerate(base.mappings_for(f[\'family_id\'])):\n                for condition in CONDITIONS:\n                    for wording in WORDINGS:\n                        prompt=diagnostic_prompt(s,condition,semantics,wording)\n                        filename=None if condition==\'text_full\' else s[\'scene_id\']+(\'__plain.png\' if condition==\'legacy_image\' else \'__ids.png\')\n                        tid=f"{s[\'scene_id\']}__map{mapping}__{condition}__{wording}"\n                        inputs.append(dict(trial_id=tid,family_id=f[\'family_id\'],stratum=f[\'stratum\'],scope=f[\'scope\'],\n                            scene_id=s[\'scene_id\'],role=s[\'role\'],mirror=s[\'mirror\'],mapping_index=mapping,\n                            semantics=list(semantics),condition=condition,wording=wording,\n                            gold_semantic=s[\'features\'][\'outcome\'],gold_label=LABELS[semantics.index(s[\'features\'][\'outcome\'])],\n                            weight_outcome=s[\'features\'][\'weight_outcome\'],prompt=prompt,\n                            prompt_sha256=hashlib.sha256(prompt.encode()).hexdigest(),image_filename=filename,\n                            image_sha256=file_hash(image_dir/filename) if filename else None,\n                            generation_audit=(f[\'family_id\']==first_per_group[f[\'stratum\']] and mapping==0),\n                            original_trial_id=None))\n    for old in snapshots[\'gemma12b\'][\'inputs\']:\n        inp=dict(old,trial_id=\'replay__\'+old[\'trial_id\'],original_trial_id=old[\'trial_id\'],\n                 scope=\'archived_balance_replay\',stratum=\'archived_balance\',wording=\'original\',generation_audit=True)\n        inputs.append(inp)\n    assert len(inputs)==600 and sum(x[\'generation_audit\'] for x in inputs)==120\n    assert len({x[\'trial_id\'] for x in inputs})==600\n    freeze(shared_root/\'inputs.json\',inputs)\n    freeze(shared_root/\'image_manifest.json\',{name:file_hash(image_dir/name) for name in sorted(p.name for p in image_dir.glob(\'*.png\'))})\n    return families,inputs,snapshots\n\n\ndef summarize_diagnostic(rows,inputs,snapshots,model_key,outdir):\n    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n    df=pd.DataFrame(rows)\n    expected={r[\'trial_id\']:r for r in inputs}\n    assert df.trial_id.is_unique and set(df.trial_id)==set(expected), \'Full analysis requires all 600 inputs.\'\n    for r in rows:\n        i=expected[r[\'trial_id\']]\n        assert r[\'gold_label\']==i[\'gold_label\'] and r[\'prompt_sha256\']==i[\'prompt_sha256\']\n        assert r[\'correct\']==int(r[\'predicted_label\']==i[\'gold_label\'])\n        assert r[\'generation_audited\']==i[\'generation_audit\']\n    df.to_json(outdir/\'trial_results.jsonl\',orient=\'records\',lines=True,force_ascii=False)\n    for side in (\'left\',\'right\',\'balanced\'):\n        df[\'answer_\'+side]=(df.predicted_semantic==side).astype(int)\n    for label in LABELS:\n        df[\'label_\'+label]=(df.predicted_label==label).astype(int)\n    metrics=[\'correct\',\'answer_left\',\'answer_right\',\'answer_balanced\',\'label_A\',\'label_B\',\'label_C\',\'gold_margin\',\'top_score_tie\']\n    fam=df.groupby([\'scope\',\'stratum\',\'family_id\',\'condition\',\'wording\'])[metrics].mean().reset_index()\n    fam.to_csv(outdir/\'family_metrics.csv\',index=False)\n    summary=fam.groupby([\'scope\',\'stratum\',\'condition\',\'wording\'])[metrics].mean().reset_index()\n    counts=fam.groupby([\'scope\',\'stratum\',\'condition\',\'wording\']).size().rename(\'n_families\').reset_index()\n    summary=summary.merge(counts)\n    summary.to_csv(outdir/\'diagnostic_summary.csv\',index=False)\n    # No overall number mixing hand-picked control groups with archive replay.\n    mirrors=[]\n    flip={\'left\':\'right\',\'right\':\'left\',\'balanced\':\'balanced\'}\n    for k,g in df.groupby([\'scope\',\'stratum\',\'family_id\',\'condition\',\'wording\',\'mapping_index\']):\n        assert len(g)==2 and set(g.mirror)=={False,True}\n        a=g[g.mirror==False].iloc[0];b=g[g.mirror==True].iloc[0]\n        mirrors.append(dict(zip([\'scope\',\'stratum\',\'family_id\',\'condition\',\'wording\',\'mapping_index\'],k),\n             both_correct=int(a.correct and b.correct),semantic_flips_as_expected=int(flip[a.predicted_semantic]==b.predicted_semantic)))\n    pd.DataFrame(mirrors).to_csv(outdir/\'mirror_pairs.csv\',index=False)\n    maps=[]\n    controls=df[df.scope==\'simple_controls\']\n    for k,g in controls.groupby([\'stratum\',\'family_id\',\'scene_id\',\'condition\',\'wording\']):\n        assert len(g)==3 and g.mapping_index.nunique()==3\n        maps.append(dict(zip([\'stratum\',\'family_id\',\'scene_id\',\'condition\',\'wording\'],k),\n                         semantic_invariant=int(g.predicted_semantic.nunique()==1),all_correct=int(g.correct.min())))\n    pd.DataFrame(maps).to_csv(outdir/\'mapping_invariance.csv\',index=False)\n    pair=[]\n    for k,g in controls.groupby([\'stratum\',\'family_id\',\'scene_id\',\'condition\',\'mapping_index\']):\n        by=g.set_index(\'wording\')\n        for name,a,b in [(\'balance_word\',\'original\',\'explicit_balance\'),(\'question_wording\',\'explicit_balance\',\'neutral_question\')]:\n            pair.append(dict(zip([\'stratum\',\'family_id\',\'scene_id\',\'condition\',\'mapping_index\'],k),comparison=name,\n                             accuracy_delta=int(by.loc[b,\'correct\'])-int(by.loc[a,\'correct\']),\n                             semantic_changed=int(by.loc[a,\'predicted_semantic\']!=by.loc[b,\'predicted_semantic\'])))\n    pd.DataFrame(pair).to_csv(outdir/\'wording_pairs.csv\',index=False)\n    generated=[]\n    for k,g in df[df.generation_audited].groupby([\'scope\',\'condition\',\'wording\']):\n        valid=g.generated_label.isin(LABELS)\n        agree=valid&(g.generated_label==g.predicted_label)\n        generated.append(dict(zip([\'scope\',\'condition\',\'wording\'],k),n=len(g),valid=int(valid.sum()),\n                              agrees=int(agree.sum()),agreement_all=float(agree.mean())))\n    pd.DataFrame(generated).to_csv(outdir/\'generation_agreement.csv\',index=False)\n    replays=[]\n    for r in rows:\n        if r[\'scope\']!=\'archived_balance_replay\':\n            continue\n        old=snapshots[model_key][\'rows\'][r[\'original_trial_id\']]\n        scores=np.array([r[\'candidate_logprobs\'][x] for x in LABELS])\n        before=np.array([old[\'candidate_logprobs\'][x] for x in LABELS])\n        replays.append(dict(trial_id=r[\'trial_id\'],original_trial_id=r[\'original_trial_id\'],condition=r[\'condition\'],\n            prompt_identical=r[\'prompt_sha256\']==old[\'prompt_sha256\'],\n            prepared_inputs_identical=r[\'prepared_inputs\']==old[\'prepared_inputs\'],\n            candidate_plan_identical=r[\'candidate_tokens\']==old[\'candidate_tokens\'] and r[\'candidate_prefix\']==old[\'candidate_prefix\'],\n            max_centered_score_difference=float(np.max(np.abs((scores-scores.mean())-(before-before.mean())))),\n            old_semantic=old[\'predicted_semantic\'],new_semantic=r[\'predicted_semantic\'],\n            same_choice=r[\'predicted_label\']==old[\'predicted_label\'],old_correct=old[\'correct\'],new_correct=r[\'correct\']))\n    pd.DataFrame(replays).to_csv(outdir/\'archived_replay_comparison.csv\',index=False)\n    return summary\n',"<embedded-diagnostic-core>","exec"),globals())
IMPLEMENTATION_SHA256='5a4cc32e046dbbcc6cd752ffcb9499f79db32a48adb86e2531a8cdc88b21c154'
BASELINE_IMPLEMENTATION='2e4dd2772d1131a18a47f3a23766fdb5985281134b88d03045a8c0261d0c3e6e'


In [ ]:
%%cell_progress 5/13 Freeze diagnostic configuration
assert MODEL_KEY in MODEL_SPECS, 'Use qwen27b or gemma12b.'
assert RUN_ID and all(c.isalnum() or c in '_-' for c in RUN_ID)
assert SOURCE_RUN_ID and all(c.isalnum() or c in '_-' for c in SOURCE_RUN_ID)
SPEC=MODEL_SPECS[MODEL_KEY]
RUN_ROOT=PROJECT_ROOT/'results/direction_balance_diagnostic_v1'/RUN_ID
SHARED_ROOT=RUN_ROOT/'shared'/PHASE
PHASE_ROOT=RUN_ROOT/'models'/MODEL_KEY/PHASE
ANALYSIS_DIR=PHASE_ROOT/'analysis'
IMAGE_DIR=SHARED_ROOT/'images'
SHARED_ROOT.mkdir(parents=True,exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True,exist_ok=True)
SOURCE_PATHS={key:Path(SOURCE_ZIPS[key]) if SOURCE_ZIPS[key] else
    PROJECT_ROOT/'results/visual_information_support_v1'/SOURCE_RUN_ID/
    f'VisualTorque_VisualInformationSupport_{key}_development_Output.zip' for key in MODEL_SPECS}
freeze(RUN_ROOT/'protocol.json',dict(schema=SCHEMA,implementation=IMPLEMENTATION_SHA256,
    baseline_implementation=BASELINE_IMPLEMENTATION,seed=SEED,models=MODEL_SPECS,
    conditions=list(CONDITIONS),wordings=list(WORDINGS),source_run_id=SOURCE_RUN_ID,
    scope='Descriptive development diagnostic; no p-values, threshold-based protocol selection or validation launch.',
    design='8 deliberate physical controls, 2 mirrors, 3 cyclic mappings, 4 input routes, 3 nested wording variants; 24 archived balance replays',
    comparisons='equal -> balanced (neither side tips) changes option text only; then neutral question changes question only',
    baselines='legacy_image vs image changes numbering, table and context together; image_full vs text_full changes image presence',
    inference='native BF16, batch1, thinking disabled, exact same parent scoring code; 120 direct generations',
    independence='two hand-chosen families per control group; mirrors and mappings are repetitions; symmetric mirrors share physical geometry',
    counts=dict(scoring=600,generation=120),old_results_read_only=True))
freeze(SHARED_ROOT/'phase_protocol.json',dict(phase=PHASE,protocol_sha256=file_hash(RUN_ROOT/'protocol.json')))
print('Model:',MODEL_KEY,'| Diagnostic output:',PHASE_ROOT)


In [ ]:
%%cell_progress 6/13 Load scoring and export helpers
"""Diagnostic-specific glue. Scoring/resume helpers are embedded from the parent notebook."""
import gc
import time
import shutil
from datetime import datetime, timezone
from tqdm.auto import tqdm
from visual_torque_toolkit.behavior.data import BehaviorExample
from visual_torque_toolkit.behavior.schema import PromptCondition


def make_example(spec):
    inp=INPUT_BY_ID[spec['trial_id']]
    scene=SCENES[inp['scene_id']]
    path=IMAGE_DIR/inp['image_filename'] if inp['image_filename'] else None
    option_text=[('balanced (neither side tips)' if semantic=='balanced' and inp['wording']!='original'
                  else SEMANTIC_TEXT[semantic]) for semantic in inp['semantics']]
    return BehaviorExample(example_id=inp['trial_id'],question_id=inp['trial_id'],scene_id=inp['scene_id'],
        condition=PromptCondition.SYMBOLIC_RULE if path is None else PromptCondition.IMAGE_RULE,
        question_kind='outcome',variant=inp['condition']+'__'+inp['wording'],split=PHASE,
        problem_type=inp['stratum'],outcome=inp['gold_semantic'],net_torque=scene['features']['net_torque'],
        prompt=inp['prompt'],image_path=path,labels=LABELS,option_text=tuple(option_text),
        option_semantics=tuple(inp['semantics']),gold_label=inp['gold_label'],gold_semantic=inp['gold_semantic'])


def init_diagnostic_runtime():
    init_runtime()
    prior=SOURCE_SNAPSHOTS[MODEL_KEY]['runtime']
    differences={}
    for name in ('model_metadata','gpu','torch','transformers','pillow','cuda','numpy','pandas',
                 'dtype','batch_size','thinking','toolkit','processor_config','image_processor_config',
                 'chat_template','tokenizer_chat_template'):
        differences[name]=dict(identical=RUNTIME_IDENTITY.get(name)==prior.get(name))
    freeze(PHASE_ROOT/'source_environment_comparison.json',differences)
    changed=[name for name,value in differences.items() if not value['identical']]
    print('Environment fields differing from the development archive:',changed or 'none')
    print('Archive replay will record input equality, score drift and choice agreement; no arbitrary score-parity assertion.')


def show_inputs():
    examples=[]
    for group in GROUPS:
        f=next(f for f in FAMILIES if f['stratum']==group)
        for mirror in (False,True):
            s=next(s for s in f['scenes'] if s['mirror']==mirror)
            filename=s['scene_id']+'__ids.png'
            with Image.open(IMAGE_DIR/filename) as im:
                examples.append((f'{group} / mirror={mirror}',im.convert('RGB').resize((448,224))))
    sheet=Image.new('RGB',(896,4*250),'white')
    draw=ImageDraw.Draw(sheet)
    for i,(name,im) in enumerate(examples):
        x,y=(i%2)*448,(i//2)*250
        sheet.paste(im,(x,y+22));draw.text((x+8,y+4),name,fill='black')
    sheet.save(SHARED_ROOT/'control_preview.png')
    display(sheet)
    print('600 scored inputs = 576 simple controls + 24 exact archived-input replays.')
    print('120 fixed direct-generation checks; no validation data or old results are modified.')
    example=next(x for x in INPUTS if x['condition']=='text_full' and x['wording']=='neutral_question')
    print(example['prompt'])


def finish_diagnostic(rows):
    summary=summarize_diagnostic(rows,INPUTS,SOURCE_SNAPSHOTS,MODEL_KEY,ANALYSIS_DIR)
    assert len(rows)==600 and sum(r['generation_audited'] for r in rows)==120
    display(summary[['scope','stratum','condition','wording','n_families','correct','answer_left','answer_balanced']].round(4))
    agreement=pd.read_csv(ANALYSIS_DIR/'generation_agreement.csv')
    display(agreement)
    replay=pd.read_csv(ANALYSIS_DIR/'archived_replay_comparison.csv')
    display(replay[['condition','prepared_inputs_identical','same_choice','max_centered_score_difference']])
    flags=dict(replay_input_mismatches=int((~replay.prepared_inputs_identical).sum()),
        replay_candidate_plan_mismatches=int((~replay.candidate_plan_identical).sum()),
        replay_changed_choices=int((~replay.same_choice).sum()),
        generation_invalid=int((agreement.n-agreement.valid).sum()),
        generation_disagreements=int((agreement.valid-agreement.agrees).sum()),
        inference_status='descriptive_diagnostic_only',validation_authorized_by_software=False)
    save_json(ANALYSIS_DIR/'measurement_checks.json',flags)
    save_json(PHASE_ROOT/'completion.json',dict(status='complete',model=MODEL_KEY,phase=PHASE,
        n_scored=600,n_generation_checks=120,implementation=IMPLEMENTATION_SHA256,
        support_sha256=file_hash(SHARED_ROOT/'support.json'),inputs_sha256=file_hash(SHARED_ROOT/'inputs.json'),
        runtime_sha256=file_hash(PHASE_ROOT/'runtime_identity.json'),
        family_files={p.name:file_hash(p) for p in sorted((PHASE_ROOT/'families').glob('*.json'))},
        scope='Simple controls and intentional archive replays; not held-out validation.',completed_at_utc=now_utc()))
    print('Diagnostic complete. Review the simple controls, wording changes and replay checks before deciding on validation.')
    return summary


def combine_diagnostics():
    frames=[]
    for key in MODEL_SPECS:
        root=RUN_ROOT/'models'/key/PHASE
        complete=root/'completion.json'
        if not complete.is_file():
            print('Combined diagnostic table pending:',key)
            return
        old=read_json(complete)
        assert old['status']=='complete' and old['implementation']==IMPLEMENTATION_SHA256
        assert old['inputs_sha256']==file_hash(SHARED_ROOT/'inputs.json')
        frames.append(pd.read_csv(root/'analysis/diagnostic_summary.csv').assign(model=key))
    combined=pd.concat(frames,ignore_index=True)
    combined.to_csv(ANALYSIS_DIR/'cross_model_diagnostic_summary.csv',index=False)
    print('Both models complete; the combined descriptive table is included in this model’s ZIP.')


def now_utc():
    return datetime.now(timezone.utc).isoformat()

def tensor_identity(inputs):
    result = {}
    for key, value in inputs.items():
        if torch.is_tensor(value):
            x = value.detach().cpu().contiguous()
            result[key] = dict(shape=list(x.shape), dtype=str(x.dtype),
                               sha256=hashlib.sha256(x.view(torch.uint8).numpy().tobytes()).hexdigest())
        else:
            result[key] = str(value)
    return result

def init_runtime():
    global RUNTIME_IDENTITY, RUNTIME_SHA
    import PIL
    import transformers
    assert adapter.metadata['resolved_revision'] == SPEC['revision']
    assert adapter.metadata['dtype'] == 'bfloat16'
    assert not getattr(adapter.model, 'is_quantized', False), 'Use native BF16, not a quantized checkpoint.'
    assert not any(str(v) in ('cpu', 'disk') for v in getattr(adapter.model, 'hf_device_map', {}).values()), 'The full BF16 model must fit on GPU.'
    adapter.model.eval()
    torch.manual_seed(SEED)
    np.random.seed(SEED % (2**32))
    torch.backends.cuda.matmul.allow_tf32 = False
    plans = {}
    for condition in CONDITIONS:
        ex = make_example(next(s for s in SPECS if s['condition'] == condition))
        plan = adapter._candidate_plan(ex)
        assert plan is not None, 'Expected exact single-token candidates after a common prefix.'
        prefix, tokens = plan
        assert set(tokens) == set(LABELS) and len(set(tokens.values())) == 3
        plans[condition] = dict(prefix=list(prefix), token_by_label=tokens)
    assert len({digest(v) for v in plans.values()}) == 1, 'Candidate scoring differs between conditions.'
    template = adapter.processor.chat_template
    tokenizer_template = adapter.processor.tokenizer.chat_template
    processor_config = adapter.processor.to_dict() if hasattr(adapter.processor, 'to_dict') else {}
    processor_config = json.loads(json.dumps(processor_config, default=str))
    image_processor = getattr(adapter.processor, 'image_processor', None)
    image_config = image_processor.to_dict() if image_processor is not None else None
    image_config = json.loads(json.dumps(image_config, default=str))
    RUNTIME_IDENTITY = dict(model_key=MODEL_KEY, model_metadata=adapter.metadata,
        gpu=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        torch=torch.__version__, transformers=transformers.__version__, pillow=PIL.__version__,
        cuda=torch.version.cuda, numpy=np.__version__, pandas=pd.__version__,
        implementation=IMPLEMENTATION_SHA256, dtype='bfloat16', batch_size=1,
        thinking=False, max_new_tokens_generation=16, candidate_plans=plans,
        toolkit={str(p.relative_to(TOOLKIT_ROOT)):file_hash(p) for p in sorted((TOOLKIT_ROOT/'src').rglob('*.py'))},
        protocol_sha256=file_hash(RUN_ROOT/'protocol.json'), phase_protocol_sha256=file_hash(SHARED_ROOT/'phase_protocol.json'),
        support_sha256=file_hash(SHARED_ROOT/'support.json'), inputs_sha256=file_hash(SHARED_ROOT/'inputs.json'),
        processor_config=processor_config, image_processor_config=image_config,
        chat_template=template, tokenizer_chat_template=tokenizer_template)
    RUNTIME_SHA = freeze(PHASE_ROOT/'runtime_identity.json', RUNTIME_IDENTITY)
    print('Frozen candidate plans:', plans['image'])
    print('Direct-answer scoring; hidden reasoning disabled. Results do not measure a reasoning-enabled ceiling.')

def score_one(spec):
    ex = make_example(spec)
    prefix, tokens = adapter._candidate_plan(ex)
    raw_inputs = adapter._template([ex.messages()], add_generation_prompt=True)
    inputs = adapter._append_text_prefix(adapter._to_device(raw_inputs), prefix)
    identity = tensor_identity(inputs)
    present_pixels = any('pixel' in k and torch.is_tensor(v) and v.numel() > 0 for k, v in inputs.items())
    assert present_pixels == (ex.image_path is not None), 'Image/text-only processor input mismatch.'
    started = time.monotonic()
    with torch.inference_mode():
        outputs = adapter.model(**inputs, use_cache=False)
        logits = outputs.logits[0, -1].float()
        logp = torch.log_softmax(logits, dim=-1)
        values = [float(logp[tokens[label]].item()) for label in LABELS]
    assert np.isfinite(values).all()
    predicted = LABELS[int(np.argmax(values))]
    semantic = ex.label_to_semantic[predicted]
    correct = int(predicted == ex.gold_label)
    gold_index = LABELS.index(ex.gold_label)
    margin = values[gold_index] - max(v for i, v in enumerate(values) if i != gold_index)
    sorted_scores = sorted(values, reverse=True)
    row = dict(spec, predicted_label=predicted, predicted_semantic=semantic, correct=correct,
        candidate_logprobs=dict(zip(LABELS, values)), gold_margin=margin,
        predicted_balance=int(semantic == 'balanced'), wrong_follows_weight=int(not correct and semantic == spec['weight_outcome']),
        top_score_tie=sorted_scores[0] == sorted_scores[1],
        prompt_sha256=hashlib.sha256(ex.prompt.encode()).hexdigest(),
        input_sha256=digest(identity), prepared_inputs=identity,
        input_token_ids=inputs['input_ids'][0].detach().cpu().tolist(),
        input_token_count=int(inputs['attention_mask'].sum()),
        scoring_mode='single_token_next_token', candidate_prefix=list(prefix), candidate_tokens=tokens,
        scoring_seconds=time.monotonic()-started, generation_audited=spec['trial_id'] in AUDIT_IDS,
        generated_label=None, raw_response=None, parsed_response=None, generated_correct=None,
        recorded_at_utc=now_utc())
    del outputs, logits, logp, inputs, raw_inputs
    if row['generation_audited']:
        generated = adapter.generate_batch([ex])[0]
        label = strict_label(generated.parsed_content)
        row.update(generated_label=label, raw_response=generated.raw_response,
                   parsed_response=generated.parsed_content,
                   generated_correct=int(label == ex.gold_label) if label is not None else None)
    return row

def run_scoring():
    folder = PHASE_ROOT/'families'
    folder.mkdir(parents=True, exist_ok=True)
    grouped = {f['family_id']: [s for s in SPECS if s['family_id'] == f['family_id']] for f in FAMILIES}
    all_rows = []
    bar = tqdm(total=len(SPECS), desc=f'{MODEL_KEY} / {PHASE}: paired inputs', unit='forward')
    try:
        for family in FAMILIES:
            fid = family['family_id']
            path = folder/(fid+'.json')
            expected = {s['trial_id']:s for s in grouped[fid]}
            rows = []
            identity = digest([RUNTIME_SHA, fid])
            if path.exists():
                old = read_json(path)
                assert old['identity'] == identity, 'Different runtime or input definition; use a new RUN_ID.'
                rows = old['rows']
                assert old['rows_sha256'] == digest(rows), 'Cached result rows changed: ' + str(path)
                assert len({r['trial_id'] for r in rows}) == len(rows)
                for row in rows:
                    assert row['trial_id'] in expected
                    assert all(row[k] == v for k, v in expected[row['trial_id']].items())
                bar.update(len(rows))
            done = {r['trial_id'] for r in rows}
            # Fixed pseudo-random ordering prevents condition/order from being confounded.
            order = sorted(expected, key=lambda key: digest(['trial_order', SEED, key]))
            try:
                for tid in order:
                    if tid in done:
                        continue
                    row = score_one(expected[tid])
                    rows.append(row)
                    bar.update(1)
                    bar.set_postfix_str(expected[tid]['condition'])
                    if len(rows) % 6 == 0:
                        save_json(path, dict(identity=identity, family_id=fid, rows=rows, rows_sha256=digest(rows)))
            finally:
                # Save completed inputs even when an error or interruption occurs.
                save_json(path, dict(identity=identity, family_id=fid, rows=rows, rows_sha256=digest(rows)))
            assert len(rows) == len(expected)
            all_rows.extend(rows)
    finally:
        bar.close()
    return all_rows

def export_support():
    """Works without model weights after the setup/definition cells have run."""
    completion_path = PHASE_ROOT/'completion.json'
    complete = completion_path.exists() and read_json(completion_path).get('status') == 'complete'
    sources = {}
    for name in ('protocol.json', 'historical_exclusion.json'):
        p = RUN_ROOT/name
        if p.exists():
            sources[name] = p
    for p in SHARED_ROOT.rglob('*'):
        if p.is_file() and not p.name.endswith('.tmp'):
            sources['shared/'+PHASE+'/'+p.relative_to(SHARED_ROOT).as_posix()] = p
    for p in PHASE_ROOT.rglob('*'):
        if p.is_file() and not p.name.endswith('.tmp'):
            sources['models/'+MODEL_KEY+'/'+PHASE+'/'+p.relative_to(PHASE_ROOT).as_posix()] = p
    manifest = dict(schema=SCHEMA, model=MODEL_KEY, phase=PHASE,
        status='complete' if complete else 'partial_not_complete',
        created_at_utc=now_utc(), files={name:file_hash(p) for name,p in sources.items()})
    output = RUN_ROOT/f'VisualTorque_Direction_Balance_Diagnostic_{MODEL_KEY}_Output.zip'
    tmp = output.with_suffix('.zip.tmp')
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, path in sources.items():
            archive.write(path, name)
        archive.writestr('export_manifest.json', json.dumps(manifest, ensure_ascii=False, indent=2))
    with zipfile.ZipFile(tmp) as archive:
        assert archive.testzip() is None
    tmp.replace(output)
    local = Path('/content')/output.name if Path('/content').is_dir() else output
    if local.resolve() != output.resolve():
        shutil.copyfile(output, local)
    print('Export status:', manifest['status'], '\nDrive:', output, '\nDownload:', local)
    try:
        from google.colab import files
        files.download(str(local))
    except ImportError:
        pass
    return str(output)


In [ ]:
%%cell_progress 7/13 Prepare simple controls and exact archived replays
cell_stage('Checking two known development ZIPs; no recursive Drive scan')
FAMILIES,INPUTS,SOURCE_SNAPSHOTS=prepare_study(SHARED_ROOT,SOURCE_PATHS,BASELINE_IMPLEMENTATION)
INPUT_BY_ID={x['trial_id']:x for x in INPUTS}
SCENES={s['scene_id']:s for f in FAMILIES for s in f['scenes']}
SPECS=[{k:v for k,v in x.items() if k not in ('prompt','image_filename','image_sha256','generation_audit')} for x in INPUTS]
AUDIT_IDS={x['trial_id'] for x in INPUTS if x['generation_audit']}
show_inputs()


In [ ]:
%%cell_progress 8/13 Check GPU and model access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select a GPU runtime that can hold the native BF16 model.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login unavailable:', type(exc).__name__)


In [ ]:
%%cell_progress 9/13 Load model and record environment
cell_stage(f'Loading native BF16 {MODEL_KEY}')
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()
adapter=HuggingFaceMultimodalAdapter(SPEC['model_id'],revision=SPEC['revision'],dtype='bfloat16',device_map='auto')
display(adapter.metadata)
init_diagnostic_runtime()


In [ ]:
%%cell_progress 10/13 Run 600 diagnostic inputs
ROWS=run_scoring()


In [ ]:
%%cell_progress 11/13 Summarize direction, balance, wording and replay
SUMMARY=finish_diagnostic(ROWS)


In [ ]:
%%cell_progress 12/13 Combine both models when available
combine_diagnostics()


In [ ]:
%%cell_progress 13/13 Export complete or partial diagnostic results
OUTPUT_ZIP=export_support()
